In [16]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
import os

model = ChatOpenAI(
    model='Qwen/Qwen3-Omni-30B-A3B-Instruct',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [22]:
# 定义提示词模板
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, FewShotChatMessagePromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ('system', '请将下面的内容翻译为{language}'),
    ('user', "{text}")
])

In [8]:
from langchain_core.output_parsers import StrOutputParser

# 定义结果解析器
parser = StrOutputParser()

In [5]:
# 定义链
chain = prompt_template | model | parser
chain

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='请将下面的内容翻译为{language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])
| ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001E96C9F7A10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E96CF386E0>, root_client=<openai.OpenAI object at 0x000001E96C8A2BA0>, root_async_client=<openai.AsyncOpenAI object at 0x000001E96CF38440>, model_name='deepseek-ai/DeepSeek-V3.2', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.siliconflow.cn/v1')
| StrOutputParser()

In [6]:
chain.invoke({
    "text": "你好，请问你要去哪里?",
    "language": "日语"
})

'こんにちは、どこへ行きますか。'

# FewShotPromptTemplate

In [4]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

# FewShotPromptTemplate
# 步骤一： 提供示例
examples = [
    {
        "question": "穆罕默德·阿里和艾伦·图灵谁活得更久？",
        "answer": """
是否需要后续问题：是。
后续问题：穆罕默德·阿里去世时多大？
中间答案：穆罕默德·阿里去世时74岁。
后续问题：艾伦·图灵去世时多大？
中间答案：艾伦·图灵去世时41岁。
所以最终答案是：穆罕默德·阿里
""",
    },
    {
        "question": "乔治·华盛顿的外祖父是谁？",
        "answer": """
是否需要后续问题：是。
后续问题：乔治·华盛顿的母亲是谁？
中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。
后续问题：玛丽·鲍尔·华盛顿的父亲是谁？
中间答案：玛丽·鲍尔·华盛顿的父亲是约瑟夫·鲍尔。
所以最终答案是：约瑟夫·鲍尔
""",
    },
    {
        "question": "《大白鲨》和《007：大战皇家赌场》的导演是否来自同一个国家？",
        "answer": """
是否需要后续问题：是。
后续问题：《大白鲨》的导演是谁？
中间答案：《大白鲨》的导演是史蒂文·斯皮尔伯格。
后续问题：史蒂文·斯皮尔伯格来自哪里？
中间答案：美国。
后续问题：《007：大战皇家赌场》的导演是谁？
中间答案：《007：大战皇家赌场》的导演是马丁·坎贝尔。
后续问题：马丁·坎贝尔来自哪里？
中间答案：新西兰。
所以最终答案是：否
""",
    },
]

base_template = PromptTemplate.from_template("问题: {question}\n{answer}")

# 步骤二：创建FewShotPromptTemplate实例
final_template = FewShotPromptTemplate(
    examples=examples,  # 传入示例列表
    example_prompt=base_template,  # 指定单个示例的提示模板
    suffix="问题: {input}",  # 最后追加的问题模板
    input_variables=["input"],  # 指定输入变量
)
final_template.invoke({"input": "中国古代历史上，唐朝和宋朝那个朝代延续最长?"})

StringPromptValue(text='问题: 穆罕默德·阿里和艾伦·图灵谁活得更久？\n\n是否需要后续问题：是。\n后续问题：穆罕默德·阿里去世时多大？\n中间答案：穆罕默德·阿里去世时74岁。\n后续问题：艾伦·图灵去世时多大？\n中间答案：艾伦·图灵去世时41岁。\n所以最终答案是：穆罕默德·阿里\n\n\n问题: 乔治·华盛顿的外祖父是谁？\n\n是否需要后续问题：是。\n后续问题：乔治·华盛顿的母亲是谁？\n中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。\n后续问题：玛丽·鲍尔·华盛顿的父亲是谁？\n中间答案：玛丽·鲍尔·华盛顿的父亲是约瑟夫·鲍尔。\n所以最终答案是：约瑟夫·鲍尔\n\n\n问题: 《大白鲨》和《007：大战皇家赌场》的导演是否来自同一个国家？\n\n是否需要后续问题：是。\n后续问题：《大白鲨》的导演是谁？\n中间答案：《大白鲨》的导演是史蒂文·斯皮尔伯格。\n后续问题：史蒂文·斯皮尔伯格来自哪里？\n中间答案：美国。\n后续问题：《007：大战皇家赌场》的导演是谁？\n中间答案：《007：大战皇家赌场》的导演是马丁·坎贝尔。\n后续问题：马丁·坎贝尔来自哪里？\n中间答案：新西兰。\n所以最终答案是：否\n\n\n问题: 中国古代历史上，唐朝和宋朝那个朝代延续最长?')

In [9]:
chain = final_template | model | parser
chain.invoke({"input": "中国古代历史上，唐朝和宋朝那个朝代延续最长?"})

'是否需要后续问题：是。  \n后续问题：唐朝存在了多少年？  \n中间答案：唐朝从公元618年到公元907年，共计289年。  \n后续问题：宋朝存在了多少年？  \n中间答案：宋朝分为北宋和南宋，从公元960年到公元1279年，共计319年。  \n所以最终答案是：宋朝'

# 聊天提示词模板

In [15]:
# 变量占位符
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个幽默的电视台主持人"),
    ("user", "帮我生成一个简短的，关于{topic}的报幕词")
])
prompt_template.invoke({"topic": "中国加油"})

ChatPromptValue(messages=[SystemMessage(content='你是一个幽默的电视台主持人', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我生成一个简短的，关于中国加油的报幕词', additional_kwargs={}, response_metadata={})])

In [18]:
# 消息占位符
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个幽默的电视台主持人"),
    MessagesPlaceholder(variable_name="input"),
])
prompt_template.invoke({"input": [HumanMessage(content="帮我生成一个简短的，关于中国加油的报幕词")]})

ChatPromptValue(messages=[SystemMessage(content='你是一个幽默的电视台主持人', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我生成一个简短的，关于中国加油的报幕词', additional_kwargs={}, response_metadata={})])

In [25]:
# ICL: few-shot
examples = [
    {"input": "2 嘎嘎 2", "output": "4"}
    , {"input": "3 嘎嘎 3", "output": "6"}
]
base_template = ChatPromptTemplate.from_messages([
    ('human', '{input}'),
    ('ai', '{output}')
])
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=base_template
)
few_shot_prompt.invoke({})

ChatPromptValue(messages=[HumanMessage(content='2 嘎嘎 2', additional_kwargs={}, response_metadata={}), AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='3 嘎嘎 3', additional_kwargs={}, response_metadata={}), AIMessage(content='6', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [30]:
final_template = ChatPromptTemplate.from_messages([
    ("system", "你是智能机器人AI助手"),
    few_shot_prompt,
    MessagesPlaceholder("msgs")
])
chain = final_template | model | parser
chain.invoke({"msgs": [HumanMessage("请计算1 嘎嘎 1")]})

'根据前两个例子：\n\n- 2 嘎嘎 2 = 4（即 2 + 2）\n- 3 嘎嘎 3 = 6（即 3 + 3）\n\n可以推测，“嘎嘎”代表的是加法操作。\n\n因此：\n\n1 嘎嘎 1 = 1 + 1 = **2**'

# 提示词模板组合

In [31]:
prompt = (
        PromptTemplate.from_template("帮我生成一个简短的，关于{topic}的报幕词") +
        ", 要求: 1、内容搞笑一些;" +
        "2、内容要 输出内容采用{language};"
)
prompt.invoke({"topic": "中国加油", "language": "English"})

StringPromptValue(text='帮我生成一个简短的，关于中国加油的报幕词, 要求: 1、内容搞笑一些;2、内容要 输出内容采用English;')